In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from catboost import CatBoostRegressor, Pool
import lightgbm as lgb
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

In [2]:
df1 = pd.read_excel(r"../india_weather_rainfall_data.xlsx")
df1.head()

,date_of_record,month,season,station_name,state,district,avg_temp,min_temp,max_temp,wind_speed,air_pressure,elevation,latitude,longitude,rainfall
0,2021-01-02,January,Winter,Gulmarg,JK,Baramulla,-2.2,-6.6,-0.8,2.2,1020.0,2652,34.05,74.4,0.1
1,2021-01-03,January,Winter,Gulmarg,JK,Baramulla,-3.6,-4.6,-1.8,3.7,1019.5,2652,34.05,74.4,4.4
2,2021-01-04,January,Winter,Gulmarg,JK,Baramulla,-3.0,-4.5,-1.1,2.1,1022.0,2652,34.05,74.4,2.3
3,2021-01-05,January,Winter,Gulmarg,JK,Baramulla,-3.3,-5.1,-1.2,2.8,1015.6,2652,34.05,74.4,35.0
4,2021-01-06,January,Winter,Gulmarg,JK,Baramulla,-3.9,-8.3,-1.0,3.4,1015.3,2652,34.05,74.4,25.5


In [23]:
df2 = df1.copy()

In [24]:
df2["date_of_record"] = pd.to_datetime(df2["date_of_record"])
df2["day_of_year"] = df2["date_of_record"].dt.dayofyear
df2["year"] = df2["date_of_record"].dt.year
df2.head(5)

,date_of_record,month,season,station_name,state,district,avg_temp,min_temp,max_temp,wind_speed,air_pressure,elevation,latitude,longitude,rainfall,day_of_year,year
0,2021-01-02,January,Winter,Gulmarg,JK,Baramulla,-2.2,-6.6,-0.8,2.2,1020.0,2652,34.05,74.4,0.1,2,2021
1,2021-01-03,January,Winter,Gulmarg,JK,Baramulla,-3.6,-4.6,-1.8,3.7,1019.5,2652,34.05,74.4,4.4,3,2021
2,2021-01-04,January,Winter,Gulmarg,JK,Baramulla,-3.0,-4.5,-1.1,2.1,1022.0,2652,34.05,74.4,2.3,4,2021
3,2021-01-05,January,Winter,Gulmarg,JK,Baramulla,-3.3,-5.1,-1.2,2.8,1015.6,2652,34.05,74.4,35.0,5,2021
4,2021-01-06,January,Winter,Gulmarg,JK,Baramulla,-3.9,-8.3,-1.0,3.4,1015.3,2652,34.05,74.4,25.5,6,2021


In [25]:
df2.isna().sum()

date_of_record         0
month                  0
season                 0
station_name           0
state                  0
district               0
avg_temp               0
min_temp           43898
max_temp          110598
wind_speed        274444
air_pressure      304664
elevation              0
latitude               0
longitude              0
rainfall          257554
day_of_year            0
year                   0
dtype: int64

In [26]:
# Keep min_temp and max_temp for target prediction, but do not use them as input features.
# This also fixes notebook state if an earlier run already dropped these columns from df2.
for temp_col in ["min_temp", "max_temp"]:
    if temp_col not in df2.columns:
        df2[temp_col] = df1[temp_col]

df2.shape

(970339, 17)

In [27]:
for i, j in zip(df2.columns, df2.isnull().sum()):
    if j:
        print(f"{i}: {round(j/df2.shape[0]*100, 3)}%")

min_temp: 4.524%
max_temp: 11.398%
wind_speed: 28.283%
air_pressure: 31.398%
rainfall: 26.543%


In [28]:
STATION_GROUP = ["state", "district", "station_name"]

df2 = df2.sort_values(["state", "district", "station_name", "date_of_record"])

df2["sin_day"] = np.sin(2 * np.pi * df2["day_of_year"] / 365.25)
df2["cos_day"] = np.cos(2 * np.pi * df2["day_of_year"] / 365.25)

for lag in [1, 2, 3, 7, 14]:
    df2[f"temp_lag_{lag}"] = df2.groupby(STATION_GROUP)["avg_temp"].shift(lag)
    df2[f"temp_max_lag_{lag}"] = df2.groupby(STATION_GROUP)["max_temp"].shift(lag)
    df2[f"rain_lag_{lag}"] = df2.groupby(STATION_GROUP)["rainfall"].shift(lag)

df2["temp_ma_3"] = (
    df2.groupby(STATION_GROUP)["avg_temp"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=2).mean())
)
df2["temp_ma_7"] = (
    df2.groupby(STATION_GROUP)["avg_temp"]
    .transform(lambda s: s.shift(1).rolling(7, min_periods=3).mean())
)
df2["temp_trend_7"] = df2["temp_lag_1"] - df2["temp_lag_7"]

for lead in range(1, 7):
    df2[f"temp_lead_{lead}"] = df2.groupby(STATION_GROUP)["avg_temp"].shift(-lead)

# Keep rows with missing weather values; the model pipeline will impute them.
df2.isna().sum()

date_of_record          0
month                   0
season                  0
station_name            0
state                   0
district                0
avg_temp                0
min_temp            43898
max_temp           110598
wind_speed         274444
air_pressure       304664
elevation               0
latitude                0
longitude               0
rainfall           257554
day_of_year             0
year                    0
sin_day                 0
cos_day                 0
temp_lag_1            406
temp_max_lag_1     111003
rain_lag_1         257960
temp_lag_2            812
temp_max_lag_2     111408
rain_lag_2         258366
temp_lag_3           1218
temp_max_lag_3     111813
rain_lag_3         258772
temp_lag_7           2842
temp_max_lag_7     113435
rain_lag_7         260396
temp_lag_14          5684
temp_max_lag_14    116277
rain_lag_14        263238
temp_ma_3             812
temp_ma_7            1218
temp_trend_7         2842
temp_lead_1           406
temp_lead_2 

In [29]:
df3 = df2.copy()
df3.isnull().sum()

date_of_record          0
month                   0
season                  0
station_name            0
state                   0
district                0
avg_temp                0
min_temp            43898
max_temp           110598
wind_speed         274444
air_pressure       304664
elevation               0
latitude                0
longitude               0
rainfall           257554
day_of_year             0
year                    0
sin_day                 0
cos_day                 0
temp_lag_1            406
temp_max_lag_1     111003
rain_lag_1         257960
temp_lag_2            812
temp_max_lag_2     111408
rain_lag_2         258366
temp_lag_3           1218
temp_max_lag_3     111813
rain_lag_3         258772
temp_lag_7           2842
temp_max_lag_7     113435
rain_lag_7         260396
temp_lag_14          5684
temp_max_lag_14    116277
rain_lag_14        263238
temp_ma_3             812
temp_ma_7            1218
temp_trend_7         2842
temp_lead_1           406
temp_lead_2 

In [30]:
cutoff_date = "2024-01-01"

train = df3[df3["date_of_record"] < cutoff_date]
test = df3[df3["date_of_record"] >= cutoff_date]

train.shape, test.shape

((805740, 43), (164599, 43))

In [31]:
feature_cols = [
    "year", "sin_day", "cos_day", "rainfall", "wind_speed", "air_pressure",
    "elevation", "latitude", "longitude", "month", "season", "state",
    "district", "station_name",
    "temp_lag_1", "temp_lag_2", "temp_lag_3", "temp_lag_7", "temp_lag_14",
    "temp_max_lag_1", "temp_max_lag_2", "temp_max_lag_3", "temp_max_lag_7", "temp_max_lag_14",
    "rain_lag_1", "rain_lag_2", "rain_lag_3", "rain_lag_7", "rain_lag_14",
    "temp_ma_3", "temp_ma_7", "temp_trend_7", "station_month_climo", "chain_temp",
]
target_cols = ["avg_temp", "temp_lead_1", "temp_lead_2", "temp_lead_3", "temp_lead_4", "temp_lead_5", "temp_lead_6"]
lead_target_cols = [c for c in target_cols if c.startswith("temp_lead_")]

len(feature_cols), len(target_cols)

(34, 7)

In [32]:
numeric_features = [
    "year", "sin_day", "cos_day", "rainfall", "wind_speed", "air_pressure",
    "elevation", "latitude", "longitude",
    "temp_lag_1", "temp_lag_2", "temp_lag_3", "temp_lag_7", "temp_lag_14",
    "temp_max_lag_1", "temp_max_lag_2", "temp_max_lag_3", "temp_max_lag_7", "temp_max_lag_14",
    "rain_lag_1", "rain_lag_2", "rain_lag_3", "rain_lag_7", "rain_lag_14",
    "temp_ma_3", "temp_ma_7", "temp_trend_7", "station_month_climo", "chain_temp",
]
categorical_features = ["month", "season", "state", "district", "station_name"]

MONTH_TO_SEASON = {
    "January": "Winter", "February": "Winter", "March": "Winter", "December": "Winter",
    "April": "Summer", "May": "Summer", "June": "Summer",
    "July": "Monsoon", "August": "Monsoon", "September": "Monsoon",
    "October": "Post-monsoon", "November": "Post-monsoon",
}

train = train.copy()
test = test.copy()

station_month_climo = (
    train.groupby(["state", "district", "station_name", "month"], observed=True)["avg_temp"]
    .mean()
    .reset_index(name="station_month_climo")
)


def horizon_from_target(target_col):
    return 0 if target_col == "avg_temp" else int(target_col.rsplit("_", 1)[-1])


def prepare_features(df, horizon, chain_temp=None):
    """Align calendar/climatology with the forecast date; optionally chain prior-step predictions."""
    base_cols = [c for c in feature_cols if c not in {"station_month_climo", "chain_temp"}]
    x = df[base_cols].copy()

    if horizon > 0:
        forecast_dates = df["date_of_record"] + pd.to_timedelta(horizon, unit="D")
        doy = forecast_dates.dt.dayofyear
        x["sin_day"] = np.sin(2 * np.pi * doy / 365.25)
        x["cos_day"] = np.cos(2 * np.pi * doy / 365.25)
        merge_month = forecast_dates.dt.month_name()
        x["month"] = merge_month
        x["season"] = merge_month.map(MONTH_TO_SEASON)
    else:
        merge_month = df["month"]

    climo_keys = df[["state", "district", "station_name"]].copy()
    climo_keys["month"] = merge_month.values
    climo_keys = climo_keys.merge(
        station_month_climo,
        on=["state", "district", "station_name", "month"],
        how="left",
    )
    x["station_month_climo"] = climo_keys["station_month_climo"].values
    x["chain_temp"] = chain_temp if chain_temp is not None else np.nan
    return x[feature_cols]


def build_model():
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), numeric_features),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
                    ]
                ),
                categorical_features,
            ),
        ]
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "lgbm",
                lgb.LGBMRegressor(
                    boosting_type="gbdt",
                    objective="regression",
                    metric="rmse",
                    num_leaves=63,
                    learning_rate=0.05,
                    feature_fraction=0.9,
                    bagging_fraction=0.8,
                    bagging_freq=5,
                    min_child_samples=40,
                    n_estimators=800,
                    random_state=42,
                    verbose=-1,
                ),
            ),
        ]
    )


def fit_and_evaluate(target_col, horizon, chain_train=None, chain_test=None):
    train_mask = train[target_col].notna()
    test_mask = test[target_col].notna()

    train_idx = train.index[train_mask]
    test_idx = test.index[test_mask]

    X_train_h = prepare_features(
        train.loc[train_mask],
        horizon,
        chain_temp=None if chain_train is None else chain_train.loc[train_idx].values,
    )
    X_test_h = prepare_features(
        test.loc[test_mask],
        horizon,
        chain_temp=None if chain_test is None else chain_test.loc[test_idx].values,
    )
    y_train = train.loc[train_mask, target_col]
    y_test = test.loc[test_mask, target_col]

    model = build_model()
    model.fit(X_train_h, y_train)
    predictions = model.predict(X_test_h)

    return model, predictions, y_test, train_idx, test_idx


models = {}
metrics = []

# Same-day average temperature (no recursive chain)
model, predictions, y_test, _, _ = fit_and_evaluate("avg_temp", horizon=0)
models["avg_temp"] = model
metrics.append(
    {
        "target": "avg_temp",
        "horizon_days": 0,
        "chained": False,
        "train_rows": int(train["avg_temp"].notna().sum()),
        "test_rows": int(test["avg_temp"].notna().sum()),
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": mean_squared_error(y_test, predictions) ** 0.5,
        "R2": r2_score(y_test, predictions),
    }
)

# Recursive multi-day forecast: lead_k uses predicted lead_{k-1} as chain_temp
chain_train = None
chain_test = None

for target_col in lead_target_cols:
    horizon = horizon_from_target(target_col)
    use_chain = horizon > 1

    model, predictions, y_test, train_idx, test_idx = fit_and_evaluate(
        target_col,
        horizon,
        chain_train=chain_train if use_chain else None,
        chain_test=chain_test if use_chain else None,
    )
    models[target_col] = model

    train_preds = pd.Series(index=train.index, dtype=float)
    train_preds.loc[train_idx] = model.predict(
        prepare_features(
            train.loc[train_idx],
            horizon,
            chain_temp=None if not use_chain else chain_train.loc[train_idx].values,
        )
    )
    test_preds = pd.Series(index=test.index, dtype=float)
    test_preds.loc[test_idx] = predictions

    chain_train = train_preds
    chain_test = test_preds

    metrics.append(
        {
            "target": target_col,
            "horizon_days": horizon,
            "chained": use_chain,
            "train_rows": int(len(train_idx)),
            "test_rows": int(len(test_idx)),
            "MAE": mean_absolute_error(y_test, predictions),
            "RMSE": mean_squared_error(y_test, predictions) ** 0.5,
            "R2": r2_score(y_test, predictions),
        }
    )

metrics_df = pd.DataFrame(metrics)
print(metrics_df)

/Users/sayangarai/Documents/Programming/Data Science/ml_env/mlenv/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['chain_temp']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/sayangarai/Documents/Programming/Data Science/ml_env/mlenv/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['chain_temp']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/sayangarai/Documents/Programming/Data Science/ml_env/mlenv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/sayangarai/Documents/Programming/Data Science/ml_env/mlenv/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without

        target  horizon_days  chained  train_rows  test_rows       MAE  \
0     avg_temp             0    False      805740     164599  0.664487   
1  temp_lead_1             1    False      805738     164195  0.960088   
2  temp_lead_2             2     True      805736     163791  1.187600   
3  temp_lead_3             3     True      805734     163387  1.310728   
4  temp_lead_4             4     True      805732     162983  1.376722   
5  temp_lead_5             5     True      805730     162579  1.408397   
6  temp_lead_6             6     True      805728     162175  1.430362   

       RMSE        R2  
0  0.925611  0.974781  
1  1.298523  0.950265  
2  1.622131  0.922204  
3  1.791865  0.904845  
4  1.874889  0.895578  
5  1.913108  0.891029  
6  1.938331  0.887890  
